In [16]:
import json

In [17]:
class Media:
    def __init__(self, title: str, type: str, genre: str, year: int, rating: float, watched: bool):
        self.title = title
        self.type = type
        self.genre = genre
        self.year = year
        self.rating = rating
        self.watched = watched
        title: str
        type: str
        genre: str
        year: int
        rating: float
        watched: bool
    def display_info(self) -> str:
        return f"""
            Title: {self.title}
            Type: {self.type}
            Genre: {self.genre}
            Year: {self.year}
            Rating: {self.rating}
            Watched: {"Yes" if self.watched else "No"}
        """


In [18]:
class MediaRepository:
    def __init__(self, path: str):
        self.path = path

    def load(self) -> list[Media]:
        try:
            with open(self.path, 'r') as file:
                data = json.load(file)
                media_list = []
                for item in data:
                    media = Media(
                        title=item['title'],
                        type=item['type'],
                        genre=item['genre'],
                        year=item['year'],
                        rating=item['rating'],
                        watched=item['watched']
                    )
                    media_list.append(media)
                return media_list
        except FileNotFoundError:
            print(f"Error: The file {self.path} was not found.")
            return []
        except json.JSONDecodeError:
            print(f"Error: The file {self.path} is not a valid JSON file.")
            return []
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return []

    def save(self, media_list: list[Media]) -> None:
        try:
            with open(self.path, 'w', encoding="utf-8") as file:
                data = []
                for media in media_list:
                    item: dict[str, str | int | float | bool] = {
                        "title": media.title,
                        "type": media.type,
                        "genre": media.genre,
                        "year": media.year,
                        "rating": media.rating,
                        "watched": media.watched,
                    }
                    data.append(item)
                json.dump(data, file, indent=4)
        except FileNotFoundError:
            print(f"Error: The file {self.path} was not found.")
            return []
        except json.JSONDecodeError:
            print(f"Error: The file {self.path} is not a valid JSON file.")
            return []
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return []

In [19]:
class TXTReportManger:
    def __init__(self):
        self.file_path = "media_report.txt"

    def create_report(self, media_list: list[Media], total_media: int, watched: int, unwatched: int, Movies: int, Series: int, Average_rating: float) -> None:
        try:
            with open(self.file_path, "w", encoding="utf-8") as f:
                f.write(f"""
                    ===== MEDIA REPORT =====

                    Total media: {total_media}
                    Watched: {watched}
                    Unwatched: {unwatched}

                    Movies: {Movies}
                    Series: {Series}

                    Average rating: {Average_rating}
                    """)
        except PermissionError:
            print(f"[Denied access]: can't access {self.file_path}")
        except Exception as e:
            print(f"[Error]: something gone wrong, error = {e}")


In [ ]:
class MediaEngine:
    def __init__(self, path: str = "media.json"):
        self.repo = MediaRepository(path)
        self.repository: list[Media] = self.repo.load()
        self.dirty: bool = False
        self.report = TXTReportManger()

    def save(self) -> None:
        self.repo.save(media_list=self.repository)
        self.dirty = False
    def add_media(self, media: Media) -> None:
        self.repository.append(media)
        self.dirty = True

    def show_all_media(self) -> list[Media]:
        return self.repository

    def show_highest_rated_media(self) -> list[Media]:
        if not self.repository:
            return []
        highest_rating = max(media.rating for media in self.repository)
        return [media for media in self.repository if media.rating == highest_rating]

    def show_watched_media(self) -> list[Media]:
        return [media for media in self.repository if media.watched]

    def show_unwatched_media(self) -> list[Media]:
        return [media for media in self.repository if not media.watched]

    def search_media(self, title: str, genre: str | None = None) -> list[Media]:
        return [media for media in self.repository if title.lower() in media.title.lower() and (genre is None or genre.lower() in media.genre.lower())]
    def remove_media(self, media:Media) -> None:
        self.repository.remove(media)
        self.dirty = True

    def update_rating(self, media:Media, new_rating:float):
        if media in self.repository:
            media.rating = new_rating
            self.dirty = True
        else:
            print(f"[Error]: Media '{media.title}' not found in the repository.")

    def mark_as_watched(self, media:Media):
        if media in self.repository:
            media.watched = True
            self.dirty = True
        else:
            print(f"[Error]: Media '{media.title}' not found in the repository.")

    def generate_report(self) -> None:
        media_list: list[Media] = self.repository
        total_media:int = len(media_list)
        watched:int = len([k for k in media_list if k.watched])
        unwatched:int = total_media - watched
        Movies:int = len([k for k in media_list if k.type == "movie"])
        Series:int = len([k for k in media_list if k.type == "series"])
        Ratings:list[float] = [k.rating for k in media_list]
        total_ratings:float = 0
        for Rating in Ratings:
            total_ratings += Rating
        Average_rating:float = (total_ratings)/total_media if total_media else 0.0
        self.report.create_report(media_list=media_list, total_media=total_media, watched=watched, unwatched=unwatched, Movies=Movies, Series=Series, Average_rating=Average_rating)

In [21]:
class MediaCLI:
    def __init__(self):
        self.engine = MediaEngine()

    def run(self):
        print("====Welcome to the Media Manager!====")
        while True:
            choice: int = int(input("""
                1 - Show All Media
                2 - Add Media
                3 - Remove Media
                4 - Update Rating
                5 - Mark as Watched
                6 - Search Media
                7 - Show Highest Rated Media
                8 - Show Watched Media
                9 - Show Unwatched Media
                10 - Exit
                CHOOSE a number 1 to 10
            """))
            match choice:
                case 1:
                    media_list:list[Media] = self.engine.show_all_media()
                    for i, media in enumerate(media_list):
                        print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")

                case 2:
                    title:str = input("Enter the title: ")
                    type:str = input("Enter the type (movie/series): ")
                    genre:str = input("Enter the genre: ")
                    year:int = int(input("Enter the year: "))
                    rating:float = float(input("Enter the rating: "))
                    watched_input = input("Have you watched it? (yes/no): ").strip().lower()
                    watched:bool = True if watched_input == "yes" else False
                    media = Media(title=title, type=type, genre=genre, year=year, rating=rating, watched=watched)
                    self.engine.add_media(media=media)
                    print(f"Media: {media.title} is added successfully!")

                case 3:
                    title:str = input("Enter the title of the media to remove: ")
                    media_list:list[Media] = self.engine.search_media(title=title)
                    if not media_list:
                        print(f"No media found with title '{title}'.")
                    else:
                        for i, media in enumerate(media_list):
                            print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")
                        index:int = int(input("Enter the number of the media to remove: ")) - 1
                        if 0 <= index < len(media_list):
                            self.engine.remove_media(media_list[index])
                            print(f"Media '{media_list[index].title}' removed successfully.")
                        else:
                            print("Invalid selection.")

                case 4:
                    title:str = input("Enter the title of the media to update rating: ")
                    media_list:list[Media] = self.engine.search_media(title=title)
                    if not media_list:
                        print(f"No media found with title '{title}'.")
                    else:
                        for i, media in enumerate(media_list):
                            print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")
                        index:int = int(input("Enter the number of the media to update rating: ")) - 1
                        if 0 <= index < len(media_list):
                            new_rating = float(input(f"Enter the new rating between 0 and 10, (old rating is = {media_list[index].rating})"))
                            self.engine.update_rating(media=media_list[index], new_rating=new_rating)
                            print(f"Media '{media_list[index].title}' rating updated successfully.")
                        else:
                            print("Invalid selection.")

                case 5:
                    title:str = input("Enter the title of the media to mark as watched: ")
                    media_list:list[Media] = self.engine.search_media(title=title)
                    if not media_list:
                        print(f"No media found with title '{title}'.")
                    else:
                        for i, media in enumerate(media_list):
                            print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")
                        index:int = int(input("Enter the number of the media to mark as watched: ")) - 1
                        if 0 <= index < len(media_list):
                            self.engine.mark_as_watched(media=media_list[index])
                            print(f"Media '{media_list[index].title}' marked as watched successfully.")
                        else:
                            print("Invalid selection.")

                case 6:
                    title = input("Enter a title: ")
                    genre = input("Enter a genre (optional): ")
                    media_list:list[Media] = self.engine.search_media(title=title, genre=genre)
                    for i, media in enumerate(media_list):
                        print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")

                case 7:
                    media_list:list[Media] = self.engine.show_highest_rated_media()
                    print("====Highest Rated Media====")
                    for i, media in enumerate(media_list):
                        print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")

                case 8:
                    media_list:list[Media] = self.engine.show_watched_media()
                    print("====Watched Media====")
                    for i, media in enumerate(media_list):
                        print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")

                case 9:
                    media_list:list[Media] = self.engine.show_unwatched_media()
                    print("====UnWatched Media====")
                    for i, media in enumerate(media_list):
                        print(f"{i + 1}. {media.title}, {media.type}, {media.rating}")

                case 10:
                    self.engine.generate_report()
                    print("Exiting...")
                    self.engine.save()
                    return

                case _:
                    print("invalid chioce!! \n choose again")

In [22]:
program: MediaCLI = MediaCLI(path="media.json")
program.run()

====Welcome to the Media Manager!====
Exiting...
